# Preprocesamiento - Consumo de energia electrica de clientes de Electro Puno S.A.A.

**Capa bronze**: descarga de los archivos crudos publicados en el portal de
Datos Abiertos del Peru (datosabiertos.gob.pe) sin ninguna transformacion.

Fuente: dataset *Consumo de energia electrica de los clientes de Electro Puno S.A.A.*
Periodo cubierto: Enero 2023 - Agosto 2026.

Este notebook solo descarga. La limpieza/normalizacion va en la capa `silver`.

In [1]:
from pathlib import Path
import hashlib
import time

import pandas as pd
import requests

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

## 1. Rutas

In [2]:
# Funciona tanto si el notebook se ejecuta desde preprocesamiento/ como desde la raiz del repo.
HERE = Path.cwd()
PREP = HERE if HERE.name == "preprocesamiento" else HERE / "preprocesamiento"
BRONZE = PREP / "bronze"
BRONZE.mkdir(parents=True, exist_ok=True)

print(f"Capa bronze: {BRONZE.resolve()}")

Capa bronze: D:\PROYECTO MAESTRIA\CURSO II - SEMESTRE II\visualizacion-datos-trabajo\preprocesamiento\bronze


## 2. Manifiesto de archivos

El portal publica nombres de archivo inconsistentes y algunos meses vienen en
`.zip` y otros en `.csv`. Normalizamos el nombre destino como
`ELPU_<YYYY-MM>.<ext>` para poder concatenar la serie despues.

Se omiten los recursos duplicados (el mismo mes publicado dos veces) y el
recurso "Ir al recurso" (Junio 2024), que no expone una URL directa.

In [3]:
FILES = [
    ("2023-01", "https://www.datosabiertos.gob.pe/sites/default/files/DatosAbiertos_%20Enero2023.csv"),
    ("2023-02", "https://www.datosabiertos.gob.pe/sites/default/files/DatosAbiertos_Febrero2023.csv"),
    ("2023-03", "https://www.datosabiertos.gob.pe/sites/default/files/DatosAbiertos_%20Marzo2023.csv"),
    ("2023-04", "https://www.datosabiertos.gob.pe/sites/default/files/DatosAbiertos_Abril2023.csv"),
    ("2023-05", "https://www.datosabiertos.gob.pe/sites/default/files/Datos%20Abiertos%20Mayo%202023.csv"),
    ("2023-06", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos_202306_0.zip"),
    ("2023-07", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos_202307.zip"),
    ("2023-08", "https://www.datosabiertos.gob.pe/sites/default/files/Datos%20abiertos%20Agosto.csv"),
    ("2023-09", "https://www.datosabiertos.gob.pe/sites/default/files/Datos%20Abiertos%20Septiembre.csv"),
    ("2023-10", "https://www.datosabiertos.gob.pe/sites/default/files/Octubre%20ELPU.csv"),
    ("2023-11", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos_202311%20%281%29.zip"),
    ("2023-12", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos_202312.zip"),
    ("2024-01", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos_202401%20%281%29.csv"),
    ("2024-02", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos_202402.zip"),
    ("2024-03", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos%20-%20Marzo%202024.csv"),
    ("2024-04", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos%20-Abril%202024.csv"),
    ("2024-05", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos%20-Mayo%202024.csv"),
    ("2024-06", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos_202406.zip"),
    ("2024-07", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos%20%20Julio2024.csv"),
    ("2024-08", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos%20%20Agosto%202024.csv"),
    ("2024-09", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos%20-%20Setiembre%202024.csv"),
    ("2024-10", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos%20-%20Octubre%202024.zip"),
    ("2024-11", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos%20Nov.zip"),
    ("2024-12", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos%20dic%202024.zip"),
    ("2025-01", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertos.zip"),
    ("2025-02", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte%20Datos%20Abiertos%2002-2025__.zip"),
    ("2025-03", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte%20Datos%20Abiertos%2003-2025.zip"),
    ("2025-04", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte%20Datos%20Abiertos%2004-2025.zip"),
    ("2025-05", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte%20Datos%20Abiertos%2005-2025.zip"),
    ("2025-06", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte%20Datos%20Abiertos%2006-2025__.zip"),
    ("2025-07", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte%20Datos%20Abiertos%2007-2025.zip"),
    ("2025-08", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte%20Datos%20Abiertos%2008-2025.zip"),
    ("2025-09", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte%20Datos%20Abiertos%2009-2025.zip"),
    ("2025-10", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte%20Datos%20Abiertos%2010-2025.zip"),
    ("2025-11", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte%20Datos%20Abiertos%2011-2025.zip"),
    ("2025-12", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte%20Datos%20Abiertos%2012-2025.zip"),
    ("2026-01", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte_Datos_Abiertos_2026_01.zip"),
    ("2026-02", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte_Datos_Abiertos_2026_02.zip"),
    ("2026-03", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte_Datos_Abiertos_2026_03.zip"),
    ("2026-04", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte_Datos_Abiertos_2026_04.zip"),
    ("2026-05", "https://www.datosabiertos.gob.pe/sites/default/files/Reporte_Datos_Abiertos_2026_05.zip"),
    ("2026-06", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertosLibro_202606.zip"),
    ("2026-07", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertosLibro_202607.zip"),
    ("2026-08", "https://www.datosabiertos.gob.pe/sites/default/files/ReporteDatosAbiertosLibro_202608.zip"),
]

# Archivos de apoyo (no son la data, pero conviene tenerlos a mano).
SIDECARS = [
    ("diccionario", "https://www.datosabiertos.gob.pe/sites/default/files/Diccionario%20de%20Datos_v2.xlsx"),
    ("metadatos", "https://www.datosabiertos.gob.pe/sites/default/files/Formato_Metadatos_ELPU.docx"),
]

print(f"{len(FILES)} archivos de datos + {len(SIDECARS)} de apoyo")
print("Rango:", FILES[0][0], "->", FILES[-1][0])

44 archivos de datos + 2 de apoyo
Rango: 2023-01 -> 2026-08


## 3. Funciones de descarga

In [4]:
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; descarga-bronze/1.0)"}


def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def download(url: str, dest: Path, session: requests.Session,
             retries: int = 3, timeout: int = 120) -> str:
    '''Descarga url -> dest. Idempotente: si el archivo ya existe, no lo baja de nuevo.'''
    if dest.exists() and dest.stat().st_size > 0:
        return "cached"

    for attempt in range(1, retries + 1):
        try:
            with session.get(url, stream=True, timeout=timeout, headers=HEADERS) as r:
                r.raise_for_status()
                tmp = dest.with_name(dest.name + ".part")
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        if chunk:
                            f.write(chunk)
                tmp.replace(dest)
            return "downloaded"
        except Exception as exc:  # noqa: BLE001
            if attempt == retries:
                return f"error: {exc}"
            time.sleep(2 * attempt)

## 4. Descarga

In [5]:
records = []
session = requests.Session()

tasks = []
for period, url in FILES:
    ext = Path(url.split("?")[0]).suffix.lower()
    tasks.append((period, url, BRONZE / f"ELPU_{period}{ext}"))
for name, url in SIDECARS:
    ext = Path(url.split("?")[0]).suffix.lower()
    tasks.append((name, url, BRONZE / f"{name}{ext}"))

for i, (period, url, dest) in enumerate(tasks, 1):
    status = download(url, dest, session)
    size = dest.stat().st_size if dest.exists() else 0
    checksum = sha256_of(dest) if size and not status.startswith("error") else ""
    records.append({
        "period": period,
        "file": dest.name,
        "status": status,
        "bytes": size,
        "sha256": checksum,
        "url": url,
    })
    flag = "OK " if not status.startswith("error") else "ERR"
    print(f"[{i:02d}/{len(tasks)}] {flag} {dest.name:<28} {size:>10,} B  {status}")

summary = pd.DataFrame(records)

[01/46] OK  ELPU_2023-01.csv             28,739,700 B  downloaded
[02/46] OK  ELPU_2023-02.csv             28,716,651 B  downloaded
[03/46] OK  ELPU_2023-03.csv             28,631,343 B  downloaded
[04/46] OK  ELPU_2023-04.csv             28,682,462 B  downloaded
[05/46] OK  ELPU_2023-05.csv             28,306,769 B  downloaded
[06/46] OK  ELPU_2023-06.zip              2,641,972 B  downloaded
[07/46] OK  ELPU_2023-07.zip              2,835,451 B  downloaded
[08/46] OK  ELPU_2023-08.csv             28,628,169 B  downloaded
[09/46] OK  ELPU_2023-09.csv             28,664,769 B  downloaded
[10/46] OK  ELPU_2023-10.csv             30,998,103 B  downloaded
[11/46] OK  ELPU_2023-11.zip              2,758,436 B  downloaded
[12/46] OK  ELPU_2023-12.zip              2,785,994 B  downloaded
[13/46] OK  ELPU_2024-01.csv             31,118,067 B  downloaded
[14/46] OK  ELPU_2024-02.zip              3,250,388 B  downloaded
[15/46] OK  ELPU_2024-03.csv             28,901,783 B  downloaded
[16/46] OK

## 5. Resumen y manifiesto

In [6]:
print(summary["status"].str.split(":").str[0].value_counts().to_string())
print(f"\nTotal descargado: {summary['bytes'].sum() / 1e6:,.1f} MB")
summary

status
downloaded    46

Total descargado: 527.2 MB


,period,file,status,bytes,sha256,url
0,2023-01,ELPU_2023-01.csv,downloaded,28739700,e1ab18f91998d22c2b0324a6ae0672d3558e14fcaab0e6...,https://www.datosabiertos.gob.pe/sites/default...
1,2023-02,ELPU_2023-02.csv,downloaded,28716651,f232f58c482fa72976b1301582f9bfaa726d72eef996a5...,https://www.datosabiertos.gob.pe/sites/default...
2,2023-03,ELPU_2023-03.csv,downloaded,28631343,3ad51276f58a9e77997ece29ded41908b6f59890d5905d...,https://www.datosabiertos.gob.pe/sites/default...
3,2023-04,ELPU_2023-04.csv,downloaded,28682462,305cdc5d0ef7d3a0dcf9d0b70f7d699ccc362054e43aad...,https://www.datosabiertos.gob.pe/sites/default...
4,2023-05,ELPU_2023-05.csv,downloaded,28306769,0d745d62a01a1aea6b0156e78629893708062a0a28efdf...,https://www.datosabiertos.gob.pe/sites/default...
5,2023-06,ELPU_2023-06.zip,downloaded,2641972,7d78e17f7a1741ee3ae903780b425a6963bb3a3fc4a12a...,https://www.datosabiertos.gob.pe/sites/default...
6,2023-07,ELPU_2023-07.zip,downloaded,2835451,c557a0a85651780620ae34711130c7cb846cdf6d814763...,https://www.datosabiertos.gob.pe/sites/default...
7,2023-08,ELPU_2023-08.csv,downloaded,28628169,a16a71987b9a25596f47a4a90712f69610fc7c3a66127a...,https://www.datosabiertos.gob.pe/sites/default...
8,2023-09,ELPU_2023-09.csv,downloaded,28664769,eb705081cedf18e71153f89ca5260568e0e76be4bf207a...,https://www.datosabiertos.gob.pe/sites/default...
9,2023-10,ELPU_2023-10.csv,downloaded,30998103,d5758484e7d312272ee527063f31f2ff4650c17ebac142...,https://www.datosabiertos.gob.pe/sites/default...


In [7]:
manifest_path = BRONZE / "_manifest.csv"
summary.to_csv(manifest_path, index=False)
print(f"Manifiesto guardado en {manifest_path}")

Manifiesto guardado en d:\PROYECTO MAESTRIA\CURSO II - SEMESTRE II\visualizacion-datos-trabajo\preprocesamiento\bronze\_manifest.csv


In [8]:
# Verificacion final: lista de archivos presentes en la capa bronze.
sorted(p.name for p in BRONZE.iterdir())

['ELPU_2023-01.csv',
 'ELPU_2023-02.csv',
 'ELPU_2023-03.csv',
 'ELPU_2023-04.csv',
 'ELPU_2023-05.csv',
 'ELPU_2023-06.zip',
 'ELPU_2023-07.zip',
 'ELPU_2023-08.csv',
 'ELPU_2023-09.csv',
 'ELPU_2023-10.csv',
 'ELPU_2023-11.zip',
 'ELPU_2023-12.zip',
 'ELPU_2024-01.csv',
 'ELPU_2024-02.zip',
 'ELPU_2024-03.csv',
 'ELPU_2024-04.csv',
 'ELPU_2024-05.csv',
 'ELPU_2024-06.zip',
 'ELPU_2024-07.csv',
 'ELPU_2024-08.csv',
 'ELPU_2024-09.csv',
 'ELPU_2024-10.zip',
 'ELPU_2024-11.zip',
 'ELPU_2024-12.zip',
 'ELPU_2025-01.zip',
 'ELPU_2025-02.zip',
 'ELPU_2025-03.zip',
 'ELPU_2025-04.zip',
 'ELPU_2025-05.zip',
 'ELPU_2025-06.zip',
 'ELPU_2025-07.zip',
 'ELPU_2025-08.zip',
 'ELPU_2025-09.zip',
 'ELPU_2025-10.zip',
 'ELPU_2025-11.zip',
 'ELPU_2025-12.zip',
 'ELPU_2026-01.zip',
 'ELPU_2026-02.zip',
 'ELPU_2026-03.zip',
 'ELPU_2026-04.zip',
 'ELPU_2026-05.zip',
 'ELPU_2026-06.zip',
 'ELPU_2026-07.zip',
 'ELPU_2026-08.zip',
 '_manifest.csv',
 'diccionario.xlsx',
 'metadatos.docx']